# Day 92: End‑to‑End Client Report Pipeline
**TechMart India** – Ingest → QA → Clean → Analyse → Report → Deliver  

This notebook builds the complete pipeline for a real‑world client report.  
Run cells sequentially. The final output is a professional Excel report.

In [16]:
import pandas as pd
import numpy as np
import datetime
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

pd.options.mode.chained_assignment = None
print("✅ Imports successful")

✅ Imports successful


## Stage 1: Data Ingestion
Load the raw Excel file, print shape, column names, and first rows.  
Store the load timestamp.

In [17]:
# Load Raw Data sheet, skip the two header rows
df_raw = pd.read_excel("Day92_ClientReportPipeline.xlsx",
                       sheet_name="Raw Data",
                       header=1)

# ★ Keep only rows with valid Order IDs starting 'TM-'
df_raw = df_raw[df_raw['Order ID'].str.startswith('TM-', na=False)].reset_index(drop=True)

load_ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

print(f"Loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns at {load_ts}")
print("\nColumn names:", list(df_raw.columns))
print("\nFirst 3 rows:")
display(df_raw.head(3))
print("\nData types:")
print(df_raw.dtypes)

Loaded: 150 rows × 10 columns at 2026-05-13 11:34

Column names: ['Order ID', 'Order Date', 'Region', 'Category', 'Product', 'Units', 'Unit Price', 'Revenue', 'Cost', 'Profit']

First 3 rows:


,Order ID,Order Date,Region,Category,Product,Units,Unit Price,Revenue,Cost,Profit
0,TM-2024-0001,2024-04-30 00:00:00,East,Fitness,Resistance Band,22.0,45000.0,990000.0,674400.0,315600.0
1,TM-2024-0002,2024-06-13 00:00:00,East,Appliances,Washing Machine,2.0,18000.0,36000.0,26900.0,9100.0
2,TM-2024-0003,2024-04-15 00:00:00,South,Appliances,Water Purifier,20.0,9000.0,180000.0,105500.0,74500.0



Data types:
Order ID       object
Order Date     object
Region         object
Category       object
Product        object
Units         float64
Unit Price    float64
Revenue       float64
Cost          float64
Profit        float64
dtype: object


## Stage 2: Data Quality Audit
Detect the 5 intentional issues in the raw data without modifying it.  
Store results in a dictionary and print them cleanly.

In [18]:
def qa_audit(df):
    qa = {}
    qa['nulls_region']    = df['Region'].isna().sum() + (df['Region'] == '').sum()
    qa['nulls_product']   = df['Product'].isna().sum() + (df['Product'] == '').sum()
    qa['duplicate_ids']   = df.duplicated(subset=['Order ID']).sum()
    qa['negative_profit'] = (df['Profit'] < 0).sum()   # any negative is anomaly
    qa['bad_dates']       = pd.to_datetime(df['Order Date'], errors='coerce').isna().sum()
    return qa

qa_report = qa_audit(df_raw.copy())

print("QA Audit Results:")
for k, v in qa_report.items():
    print(f"  {k}: {v} issue(s) found")
print(f"\nTotal unique issues: {sum(qa_report.values())}")

QA Audit Results:
  nulls_region: 1 issue(s) found
  nulls_product: 1 issue(s) found
  duplicate_ids: 1 issue(s) found
  negative_profit: 1 issue(s) found
  bad_dates: 1 issue(s) found

Total unique issues: 5


## Stage 3: Data Cleaning
Apply all 5 fixes to a copy of the raw DataFrame.  
Return the cleaned DataFrame and a list of fix descriptions.

In [19]:
def clean_pipeline(df, qa_report):
    df = df.copy()
    fixes_log = []

    # Fix 1: Missing Region → 'Unknown'
    mask_region = df['Region'].isna() | (df['Region'] == '')
    df.loc[mask_region, 'Region'] = 'Unknown'
    fixes_log.append(f"Fix 1: {qa_report['nulls_region']} null Region → 'Unknown'")

    # Fix 2: Drop duplicate Order IDs (keep first)
    initial_rows = len(df)
    df = df.drop_duplicates(subset=['Order ID'], keep='first').reset_index(drop=True)
    fixes_log.append(f"Fix 2: {initial_rows - len(df)} duplicate Order IDs removed")

    # Fix 3: Negative profit → 0
    mask_profit = df['Profit'] < 0
    fixes_log.append(f"Fix 3: {mask_profit.sum()} negative profit rows → 0")
    df.loc[mask_profit, 'Profit'] = 0

    # Fix 4: Bad dates → NaT → fill with median
    df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
    median_date = df['Order Date'].dropna().median()
    df['Order Date'] = df['Order Date'].fillna(median_date)
    fixes_log.append(f"Fix 4: {qa_report['bad_dates']} bad dates → filled with median {median_date.date()}")

    # Fix 5: Missing Product → 'Unlisted'
    mask_product = df['Product'].isna() | (df['Product'] == '')
    df.loc[mask_product, 'Product'] = 'Unlisted'
    fixes_log.append(f"Fix 5: {qa_report['nulls_product']} null Product → 'Unlisted'")

    return df, fixes_log

df_clean, fixes_log = clean_pipeline(df_raw.copy(), qa_report)

print("✅ Cleaning complete.")
print(f"Shape before: {df_raw.shape}, after: {df_clean.shape}")
print("\nFixes applied:")
for fix in fixes_log:
    print(f"  • {fix}")

✅ Cleaning complete.
Shape before: (150, 10), after: (149, 10)

Fixes applied:
  • Fix 1: 1 null Region → 'Unknown'
  • Fix 2: 1 duplicate Order IDs removed
  • Fix 3: 1 negative profit rows → 0
  • Fix 4: 1 bad dates → filled with median 2024-03-26
  • Fix 5: 1 null Product → 'Unlisted'


## Stage 4: Analysis Layer
Run the four required aggregations on the clean data.  
Store each result in a dedicated DataFrame.

In [20]:
# a) Revenue by Region
df_region = (df_clean.groupby('Region')['Revenue'].sum()
                     .sort_values(ascending=False).reset_index())
df_region.columns = ['Region', 'Total Revenue']

# b) Profit Margin % by Category
df_cat = (df_clean.groupby('Category').agg(Revenue=('Revenue','sum'), Profit=('Profit','sum'))
                  .reset_index())
df_cat['Profit Margin %'] = (df_cat['Profit'] / df_cat['Revenue'] * 100).round(2)
df_category = df_cat.sort_values('Profit Margin %', ascending=False)
df_category = df_category[['Category', 'Revenue', 'Profit', 'Profit Margin %']]

# c) Top 5 Products by Revenue
df_top5 = (df_clean.groupby('Product')['Revenue'].sum()
                  .sort_values(ascending=False).head(5).reset_index())
df_top5.columns = ['Product', 'Total Revenue']
df_top5.insert(0, 'Rank', range(1, 6))

# d) Monthly Revenue Trend
df_clean['Month'] = df_clean['Order Date'].dt.to_period('M').astype(str)
df_monthly = (df_clean.groupby('Month')['Revenue'].sum().reset_index())
df_monthly.columns = ['Month', 'Revenue']
df_monthly = df_monthly.sort_values('Month')

print("Revenue by Region:"); display(df_region)
print("Category Margins:"); display(df_category)
print("Top 5 Products:"); display(df_top5)
print("Monthly Trend:"); display(df_monthly)

Revenue by Region:


,Region,Total Revenue
0,East,7426900.0
1,South,6724500.0
2,North,5690600.0
3,West,3924000.0
4,Unknown,207000.0


Category Margins:


,Category,Revenue,Profit,Profit Margin %
1,Electronics,4902500.0,1667800.0,34.02
3,Furniture,3407000.0,1151200.0,33.79
2,Fitness,7821900.0,2527500.0,32.31
0,Appliances,7590000.0,2221500.0,29.27
4,Nutrition,251600.0,65300.0,25.95


Top 5 Products:


,Rank,Product,Total Revenue
0,1,Resistance Band,3416000.0
1,2,Refrigerator,2674000.0
2,3,Water Purifier,2336000.0
3,4,Dumbbells,1968000.0
4,5,Smart TV,1677500.0


Monthly Trend:


,Month,Revenue
0,2024-01,4211000.0
1,2024-02,2672300.0
2,2024-03,5899200.0
3,2024-04,3591200.0
4,2024-05,4422500.0
5,2024-06,3176800.0


## Stage 5: Report Generation
Write a 7‑sheet Excel workbook using openpyxl.  
Includes the bonus ‘Data Story’ sheet.  
All numbers are formatted and the executive summary uses NRA formatting.

In [21]:
def write_report(df_region, df_category, df_top5, df_monthly, qa_report, fixes_log, df_clean):
    wb = Workbook()

    header_font = Font(bold=True, color="FFFFFF")
    header_fill = PatternFill(start_color="000080", end_color="000080", fill_type="solid")
    header_align = Alignment(horizontal="center", vertical="center")
    thin_border = Border(left=Side(style='thin'), right=Side(style='thin'),
                         top=Side(style='thin'), bottom=Side(style='thin'))
    num_fmt = '#,##0'

    def style_header(ws, row, num_cols):
        for col in range(1, num_cols+1):
            cell = ws.cell(row=row, column=col)
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = header_align
            cell.border = thin_border

    # Sheet 1: QA Report
    ws_qa = wb.active
    ws_qa.title = "QA Report"
    ws_qa.append(["Issue Type", "Count"])
    for k,v in qa_report.items():
        ws_qa.append([k, v])
    ws_qa.append([])
    ws_qa.append(["Fix Applied", "Description"])
    for fix in fixes_log:
        ws_qa.append(["", fix])
    style_header(ws_qa, 1, 2)
    style_header(ws_qa, len(qa_report)+3, 2)
    ws_qa.column_dimensions['A'].width = 22
    ws_qa.column_dimensions['B'].width = 65

    # Sheet 2: Revenue by Region
    ws_reg = wb.create_sheet("Revenue by Region")
    ws_reg.append(list(df_region.columns))
    for _, row in df_region.iterrows():
        ws_reg.append(list(row))
    ws_reg.append(["Total", df_region['Total Revenue'].sum()])
    style_header(ws_reg, 1, len(df_region.columns))
    ws_reg.column_dimensions['A'].width = 18
    ws_reg.column_dimensions['B'].width = 20
    for r in ws_reg.iter_rows(min_row=2, max_col=2, max_row=ws_reg.max_row):
        for c in r:
            c.number_format = num_fmt
            c.border = thin_border

    # Sheet 3: Category Margins
    ws_cat = wb.create_sheet("Category Margins")
    ws_cat.append(['Category','Revenue','Profit','Profit Margin %'])
    for _, row in df_category.iterrows():
        ws_cat.append([row['Category'], row['Revenue'], row['Profit'], row['Profit Margin %']])
    style_header(ws_cat, 1, 4)
    green = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
    red   = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    for r in range(2, ws_cat.max_row+1):
        cell = ws_cat.cell(row=r, column=4)
        cell.number_format = '0.00'
        if isinstance(cell.value, (int,float)):
            if cell.value > 25: cell.fill = green
            elif cell.value < 15: cell.fill = red
    ws_cat.column_dimensions['A'].width = 16
    ws_cat.column_dimensions['B'].width = 15
    ws_cat.column_dimensions['C'].width = 15
    ws_cat.column_dimensions['D'].width = 18

    # Sheet 4: Top 5 Products
    ws_top = wb.create_sheet("Top 5 Products")
    ws_top.append(['Rank','Product','Total Revenue'])
    for _, row in df_top5.iterrows():
        ws_top.append([int(row['Rank']), row['Product'], row['Total Revenue']])
    style_header(ws_top, 1, 3)
    ws_top.column_dimensions['A'].width = 8
    ws_top.column_dimensions['B'].width = 30
    ws_top.column_dimensions['C'].width = 20
    for r in ws_top.iter_rows(min_row=2, max_col=3, max_row=6):
        for c in r:
            c.border = thin_border
            if c.column == 3: c.number_format = num_fmt

    # Sheet 5: Monthly Trend
    ws_mon = wb.create_sheet("Monthly Trend")
    ws_mon.append(['Month','Revenue'])
    for _, row in df_monthly.iterrows():
        ws_mon.append([row['Month'], row['Revenue']])
    style_header(ws_mon, 1, 2)
    ws_mon.column_dimensions['A'].width = 14
    ws_mon.column_dimensions['B'].width = 20
    for r in ws_mon.iter_rows(min_row=2, max_col=2, max_row=ws_mon.max_row):
        for c in r:
            c.number_format = num_fmt
            c.border = thin_border

    # Sheet 6: Executive Summary (NRA)
    ws_exec = wb.create_sheet("Executive Summary")
    ws_exec.merge_cells('A1:D1')
    ws_exec.cell(row=1, column=1).value = f"TechMart India — Executive Summary ({datetime.date.today().strftime('%d-%b-%Y')})"
    ws_exec.cell(row=1, column=1).font = Font(bold=True, size=14)

    top_region = df_region.iloc[0]
    second_region = df_region.iloc[1] if len(df_region)>1 else None
    top_cat_margin = df_category.iloc[0]
    lowest_cat_margin = df_category.iloc[-1]
    top_product = df_top5.iloc[0]
    total_revenue = df_clean['Revenue'].sum()
    top_region_cat = df_clean[df_clean['Region']==top_region['Region']].groupby('Category')['Revenue'].sum().idxmax()

    bullets = [
        ("Revenue Leadership",
         f"{top_region['Region']} generated ₹{top_region['Total Revenue']:,.0f} in revenue — "
         f"{((top_region['Total Revenue']/second_region['Total Revenue']-1)*100):.0f}% above the second‑highest region." if second_region is not None else "",
         f"Driven by {top_region_cat} category dominance.",
         f"Replicate {top_region['Region']}’s category mix in the lowest performing region to close the revenue gap."),
        ("Margin Alert",
         f"{top_cat_margin['Category']} delivered {top_cat_margin['Profit Margin %']:.1f}% profit margin vs. {lowest_cat_margin['Profit Margin %']:.1f}% for {lowest_cat_margin['Category']}.",
         f"{lowest_cat_margin['Category']} may have higher return rates, discounting pressure, or cost inefficiencies.",
         f"Audit pricing and procurement for {lowest_cat_margin['Category']} before next quarter."),
        ("Product Concentration Risk",
         f"{top_product['Product']} alone generated ₹{top_product['Total Revenue']:,.0f} — {top_product['Total Revenue']/total_revenue*100:.1f}% of total company revenue.",
         "Single‑product dependency creates vulnerability if supply chain or demand shifts.",
         f"Develop a cross‑sell programme to expand buyers of {top_product['Product']} into adjacent categories.")
    ]

    row = 3
    for title, n, r, a in bullets:
        ws_exec.cell(row=row, column=1).value = title
        ws_exec.cell(row=row, column=1).font = Font(bold=True, underline="single")
        row += 1
        ws_exec.cell(row=row, column=1).value = n
        ws_exec.cell(row=row, column=1).font = Font(bold=True)
        row += 1
        ws_exec.cell(row=row, column=1).value = r
        ws_exec.cell(row=row, column=1).font = Font(italic=True)
        row += 1
        ws_exec.cell(row=row, column=1).value = a
        ws_exec.cell(row=row, column=1).font = Font(underline="single")
        row += 2
    ws_exec.column_dimensions['A'].width = 120

    # Sheet 7 (BONUS): Data Story
    ws_story = wb.create_sheet("Data Story")
    ws_story.merge_cells('A1:D1')
    ws_story.cell(row=1, column=1).value = "TechMart H1 2024 Performance Narrative"
    ws_story.cell(row=1, column=1).font = Font(bold=True, size=13)

    story = (
        f"TechMart India closed H1 2024 with total revenue of ₹{total_revenue:,.0f} across {len(df_clean)} cleaned transactions. "
        f"The {top_region['Region']} region emerged as the clear leader, contributing ₹{top_region['Total Revenue']:,.0f} "
        f"or {top_region['Total Revenue']/total_revenue*100:.1f}% of total sales, powered by {top_region_cat} products. "
        f"Profitability is strongest in {top_cat_margin['Category']} (margin {top_cat_margin['Profit Margin %']:.1f}%), "
        f"while {lowest_cat_margin['Category']} lags at {lowest_cat_margin['Profit Margin %']:.1f}%. "
        f"The product portfolio shows concentration risk: the top 5 products drive a disproportionate share of revenue, "
        f"with {top_product['Product']} alone accounting for {top_product['Total Revenue']/total_revenue*100:.1f}%. "
        f"Moving into H2, the data suggests two high‑impact actions: intensify Appliance promotion in underperforming regions, "
        f"and launch a cross‑sell initiative around the highest‑grossing products to reduce single‑product dependency."
    )
    ws_story.cell(row=3, column=1).value = story
    ws_story.column_dimensions['A'].width = 120
    ws_story.row_dimensions[3].height = 120
    ws_story.cell(row=3, column=1).alignment = Alignment(wrap_text=True, vertical="top")

    output_path = "TechMart_Report_Day92.xlsx"
    wb.save(output_path)
    return output_path

output_file = write_report(df_region, df_category, df_top5, df_monthly,
                           qa_report, fixes_log, df_clean)
print(f"✅ Report saved as: {output_file}")

✅ Report saved as: TechMart_Report_Day92.xlsx


In [7]:
# Generate the report
output_file = write_report(df_region, df_category, df_top5, df_monthly,
                           qa_report, fixes_log, df_clean)
print(f"✅ Report saved as: {output_file}")

✅ Report saved as: TechMart_Report_Day92.xlsx


## Stage 6: Delivery Confirmation
Print a clean summary to confirm the pipeline ran end‑to‑end.

In [22]:
print("=" * 55)
print("TECHMART INDIA — PIPELINE DELIVERY LOG")
print("=" * 55)
print(f"Source file    : Day92_ClientReportPipeline.xlsx")
print(f"Rows loaded    : {len(df_raw)}")
print(f"Rows after QA  : {len(df_clean)}")
print("QA fixes applied:")
for fix in fixes_log:
    print(f"   • {fix}")
print(f"Output file    : {output_file}")
print(f"Generated at   : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 55)
print("✅ READY FOR CLIENT DELIVERY")

TECHMART INDIA — PIPELINE DELIVERY LOG
Source file    : Day92_ClientReportPipeline.xlsx
Rows loaded    : 150
Rows after QA  : 149
QA fixes applied:
   • Fix 1: 1 null Region → 'Unknown'
   • Fix 2: 1 duplicate Order IDs removed
   • Fix 3: 1 negative profit rows → 0
   • Fix 4: 1 bad dates → filled with median 2024-03-26
   • Fix 5: 1 null Product → 'Unlisted'
Output file    : TechMart_Report_Day92.xlsx
Generated at   : 2026-05-13 11:35
✅ READY FOR CLIENT DELIVERY
